In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy
import math
import random
import os

from collections import deque

In [2]:
ATTACK_START = False

In [3]:
ATTACK_TYPE = 'label'
K_FACTOR = 32

In [4]:
checkpoint_path = f'./save-{ATTACK_TYPE}-{K_FACTOR}'

In [5]:
os.makedirs(checkpoint_path, exist_ok=True)

In [6]:
batch_size = 256
l_rate = 0.001

In [7]:
@tf.keras.utils.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self, trainable=True, dtype=None, **kwargs):
        super().__init__(trainable=trainable, dtype=dtype, **kwargs)
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def get_config(self):
        config = super().get_config()
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [8]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=l_rate)
        self.freqs = {}
        test_dataset = pd.read_csv("Test.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X_v_all = test_dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32) 
        self.y_v_all = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

        test_dataset_24 = test_dataset[test_dataset['freq'] == 2.4]
        test_dataset_25 = test_dataset[test_dataset['freq'] == 2.5]
        test_dataset_26 = test_dataset[test_dataset['freq'] == 2.6]
        self.X_v = {}
        self.y_v = {}
        self.X_v[2.4] = test_dataset_24.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y_v[2.4] = test_dataset_24.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.X_v[2.5] = test_dataset_25.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y_v[2.5] = test_dataset_25.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.X_v[2.6] = test_dataset_26.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y_v[2.6] = test_dataset_26.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        
        self.r2sv = []
        self.model(self.X_v_all)

        self.node_mse_his = {}
        self.node_grad_norm_his = {}


        self.node_A = {2.4: 0, 2.5: 0, 2.6: 0}
        self.node_D = {2.4: 0, 2.5: 0, 2.6: 0}
        
        self.Dic_A = {2.4: 'A', 2.5: 'B', 2.6: 'C'}
        self.Dic_D = {2.4: 'a', 2.5: 'b', 2.6: 'c'}

        self.TP = 0
        self.TN = 0
        self.FP = 0
        self.FN = 0

    def upload(self, grads, freq, GROUNDTRUTH):
        global ATTACK_START
        checkpoint = tf.train.Checkpoint(model=self.model, optimizer=self.optimizer)
        save_path = checkpoint.save(checkpoint_path)

        # 更新
        self.optimizer.apply_gradients(grads_and_vars=zip(grads, self.model.variables))

        after_y_v = self.model(self.X_v_all)
        after_mse = tf.reduce_mean(tf.square(after_y_v - self.y_v_all))

        poision_judge = self.check_anoymous(after_mse, grads, freq)

        if GROUNDTRUTH:
            ATTACK_START = True
            # checkpoint.restore(save_path)
        if ATTACK_START:
            if GROUNDTRUTH:
                if poision_judge:
                    self.TP += 1
                    self.node_D[freq] += 1
                else:
                    self.FN += 1
                    self.node_A[freq] += 1
            else:
                if poision_judge:
                    self.FP += 1
                    self.node_D[freq] += 1
                else:
                    self.TN += 1
                    self.node_A[freq] += 1
                
        cur_y_v = self.model(self.X_v[freq])
        score = 1 - tf.reduce_sum(tf.square(cur_y_v - self.y_v[freq])) / tf.reduce_sum(tf.square(self.y_v[freq] - tf.reduce_mean(self.y_v[freq])))

        self.freqs[freq] = max(0, score)

        return self.model, self.freqs
    
    def download(self):
        return self.model, self.freqs

    def lr_decay(self, ratio):
        self.optimizer.learning_rate = self.optimizer.learning_rate * ratio

    def check_anoymous(self, cur_mse, cur_grad, nodeId):
        # 检查MSE异常
        if nodeId not in self.node_mse_his:
            self.node_mse_his[nodeId] = deque(maxlen=16)
        if len(self.node_mse_his[nodeId]) < 16:
            is_mse_anomaly = False
        else:
            mse_mean = np.mean(self.node_mse_his[nodeId])
            mse_std = np.std(self.node_mse_his[nodeId])
            mse_threshold = mse_mean + 3 * mse_std
            is_mse_anomaly = cur_mse > mse_threshold
        

        # 检查梯度异常
        grad_vector = tf.concat([tf.reshape(g, [-1]) for g in cur_grad], axis=0)
        grad_norm = np.linalg.norm(grad_vector)
        if nodeId not in self.node_grad_norm_his:
            self.node_grad_norm_his[nodeId] = deque(maxlen=16)

        if len(self.node_grad_norm_his[nodeId]) < 16:
            is_grad_anomaly = False
        else:
            grad_mean = np.mean(self.node_grad_norm_his[nodeId])
            grad_std = np.std(self.node_grad_norm_his[nodeId])
            grad_threshold = grad_mean + 3 * grad_std
            is_grad_anomaly = grad_norm > grad_threshold

        if is_mse_anomaly and is_grad_anomaly:
            return True
        else:
            self.node_mse_his[nodeId].append(cur_mse)
            self.node_grad_norm_his[nodeId].append(grad_norm)
            return False

In [9]:
r2s = {2.4:[],2.5:[],2.6:[]}

In [10]:
ps = ParaServer()

In [11]:
class Node:
    def __init__(self, dsName, freq):
        self.freq = freq
        self.otfreqs = {}
        self.model = MLP()
        self.zeroModel = MLP()
        self.dataset = pd.read_csv(dsName, encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X = self.dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y = self.dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=self.X.shape[0])
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
        self.iterator = iter(self.dataset_train)

        self.current_epoch = 0
        self.epoch_r2 = []
        self.epoch_mse = []
        self.epoch_rmse = []
        self.epoch_mae = []
        self.batch_count = 0

    def getZero(self):
        m, freqs = ps.download()
        self.otfreqs = copy.deepcopy(freqs)
        self.zeroModel = copy.deepcopy(m)
        # print(self.otfreqs)
    def train(self, index_round):
        poisioning = False
        if index_round > 300 * 112 and random.random() < 0.01:
            poisioning = True
            # print('*',end="")
            poisoned_indices = np.random.choice(len(self.X), size=batch_size, replace=False)
            posionedX = self.X[poisoned_indices].copy()
            posionedY = self.y[poisoned_indices].copy()
            if ATTACK_TYPE == 'label':
                epsilon = np.random.normal(0, np.std(posionedY, 0) * K_FACTOR, posionedY.shape)
                posionedY += epsilon
            elif ATTACK_TYPE == 'feature':
                epsilon = np.random.normal(0, np.std(posionedX[:,1:], 0) * K_FACTOR, posionedX[:,1:].shape)
                posionedX[:,1:] += epsilon
            X, y = tf.convert_to_tensor(posionedX, dtype=tf.float32), tf.convert_to_tensor(posionedY, dtype=tf.float32)
        else:
            # print('-', end="")
            try:
                X, y = next(self.iterator)
                    
            except StopIteration:
                avg_r2 = np.mean(self.epoch_r2)
                avg_mse = np.mean(self.epoch_mse)
                avg_mae = np.mean(self.epoch_mae)
                avg_rmse = np.mean(self.epoch_rmse)
                print(f"\nNode {self.freq} Epoch {self.current_epoch} Summary:")
                print(f"  MSE: {avg_mse:.4f} | RMSE: {avg_rmse:.4f}")
                print(f"  MAE: {avg_mae:.4f} | R²: {avg_r2:.4f}")
                print(f"  Anomaly Detection: {ps.node_A[self.freq]}/{ps.node_D[self.freq]}")
                print(f"  {ps.TP} | {ps.FN}")
                print(f"  {ps.FP} | {ps.TN}")
                r2s[self.freq].append(avg_r2)
                ps.node_A[self.freq], ps.node_D[self.freq] = 0, 0

                self.epoch_r2 = []
                self.epoch_mse = []
                self.epoch_mae = []
                self.epoch_rmse = []
                self.current_epoch += 1
                self.batch_count = 0

                self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
                self.dataset_train = self.dataset_train.shuffle(buffer_size=self.X.shape[0])
                self.dataset_train = self.dataset_train.batch(batch_size)
                self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
                self.iterator = iter(self.dataset_train)
                X, y = next(self.iterator)

        self.model, _ = ps.download()

        with tf.GradientTape() as tape:
            y_pred = self.model(X)
            tr_mse = tf.reduce_mean(tf.square(y_pred - y))

        tr_rmse = tf.sqrt(tr_mse)
        tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
        tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(tf.square(y - tf.reduce_mean(y)))

        grads = tape.gradient(tr_mse, self.model.variables)

        sum_r2 = 1
        for k, v in self.otfreqs.items():
            if math.isclose(k, self.freq) or math.isclose(v, 0):
                continue
            X_i = tf.tensor_scatter_nd_update(X, [[i, 0] for i in range(X.shape[0])], [k] * X.shape[0])
            y_i = self.zeroModel(X_i)
            with tf.GradientTape() as tape:
                y_pred_i = self.model(X_i)
                loss = tf.reduce_mean(tf.square(y_pred_i - y_i))
            grad = tape.gradient(loss, self.model.variables)
            grads = [grads[i] + grad[i] * v for i in range(len(grads))]
            sum_r2 += v
        self.model, _ = ps.upload([i / sum_r2 for i in grads], self.freq, poisioning)

        # if epoch_index in np.arange(0, num_epochs, 25).tolist() or epoch_index == num_epochs - 1:
        self.epoch_r2.append(tr_r2)
        self.epoch_mse.append(tr_mse)
        self.epoch_mae.append(tr_mae)
        self.epoch_rmse.append(tr_rmse)

In [12]:
nodeList = [Node('./24Train.csv', 2.4), Node('./25Train.csv', 2.5), Node('./26Train.csv', 2.6)]

In [13]:
turn = [np.array([[26, 104], [178, 312], [344, 464], [520, 600]]) * 112,
        np.array([[0, 94], [149, 223], [319, 433], [464, 580]]) * 112,
        np.array([[32, 151], [155, 248], [270, 354], [378, 502]]) * 112]

In [14]:
orders = [0, 1, 2]

for i in range(600 * 112):
    if i % 112 == 0:
        print(f"EPOCH: {i // 112}")
    random.shuffle(orders)
    for j in orders:
        for l, r in turn[j]:
            if l <= i < r:
                if math.isclose(l, i):
                    nodeList[j].getZero()
                nodeList[j].train(i)
    # valiAll(i)
    if i in [100 * 112, 200 * 112, 300 * 112, 400 * 112, 500 * 112]:
        ps.lr_decay(0.7)

EPOCH: 0
EPOCH: 1

Node 2.5 Epoch 0 Summary:
  MSE: 1.4736 | RMSE: 0.6045
  MAE: 0.4973 | R²: -11.3173
  Anomaly Detection: 0/0
  0 | 0
  0 | 0
EPOCH: 2

Node 2.5 Epoch 1 Summary:
  MSE: 0.0845 | RMSE: 0.2906
  MAE: 0.2362 | R²: 0.2993
  Anomaly Detection: 0/0
  0 | 0
  0 | 0
EPOCH: 3

Node 2.5 Epoch 2 Summary:
  MSE: 0.0830 | RMSE: 0.2880
  MAE: 0.2327 | R²: 0.3116
  Anomaly Detection: 0/0
  0 | 0
  0 | 0
EPOCH: 4

Node 2.5 Epoch 3 Summary:
  MSE: 0.0811 | RMSE: 0.2846
  MAE: 0.2293 | R²: 0.3277
  Anomaly Detection: 0/0
  0 | 0
  0 | 0
EPOCH: 5

Node 2.5 Epoch 4 Summary:
  MSE: 0.0758 | RMSE: 0.2752
  MAE: 0.2213 | R²: 0.3716
  Anomaly Detection: 0/0
  0 | 0
  0 | 0
EPOCH: 6

Node 2.5 Epoch 5 Summary:
  MSE: 0.0744 | RMSE: 0.2726
  MAE: 0.2190 | R²: 0.3830
  Anomaly Detection: 0/0
  0 | 0
  0 | 0
EPOCH: 7

Node 2.5 Epoch 6 Summary:
  MSE: 0.0718 | RMSE: 0.2680
  MAE: 0.2149 | R²: 0.4041
  Anomaly Detection: 0/0
  0 | 0
  0 | 0
EPOCH: 8

Node 2.5 Epoch 7 Summary:
  MSE: 0.0706 | RMSE: 

In [15]:
print(f"  {ps.TP} | {ps.FN}")
print(f"  {ps.FP} | {ps.TN}")

  375 | 297
  2404 | 66283


In [16]:
Accuracy = (ps.TP + ps.TN) / (ps.TP + ps.TN + ps.FP + ps.FN)

In [17]:
Recall = ps.TP / (ps.TP + ps.FN)

In [18]:
Precision = ps.TP / (ps.TP + ps.FP)

In [19]:
f1_score = 2 * Precision * Recall / (Precision + Recall)

In [20]:
Accuracy

0.9610576853760867

In [21]:
Recall

0.5580357142857143

In [22]:
Precision

0.13494062612450522

In [23]:
f1_score

0.21732831063459868

In [24]:
y_v_p = ps.model(ps.X_v_all)
mse = tf.reduce_mean(tf.square(y_v_p - ps.y_v_all), axis=-1)

In [25]:
np.save(f'UnPro_{ATTACK_TYPE}-{K_FACTOR}.npy', mse.numpy())